# Controlling the Output - Key Parameters

Hello and welcome back. We know how to send a prompt and get a response, but to truly engineer our outputs, we need to learn how to control the generation process itself. This is done by using optional parameters in our API calls.

These parameters are the control knobs for the AI. They let us fine-tune everything from creativity and length to format and stopping points. This notebook will provide a hands-on demonstration of the most important ones.

## Setup

Let's import the necessary libraries and define a few variables to avoid duplicating information.

In [6]:
import json
import litellm
from dotenv import load_dotenv

load_dotenv()

MODEL_NAME = "ollama/qwen2.5:3b"

def get_completion(messages, **kwargs):
    if "model" not in kwargs:
        kwargs["model"] = MODEL_NAME
        
    return litellm.completion(
        messages=messages,
        **kwargs
    )

## `temperature` - The Creativity Knob

`temperature` controls the randomness of the output. A low temperature (for example, 0.1) is deterministic and good for factual tasks. A high temperature (for example, 1.5) is creative and good for brainstorming.

Let's ask the model to brainstorm a name for a new Python testing library.

In [10]:
TESTING_LIBRARY_NAME = [
    {
        "role":"user",
        "content": "Brainstorm a creative name for a new Python library"
    }
]

first_completion = get_completion(
    TESTING_LIBRARY_NAME, 
    temperature=0.1
).choices[0].message.content

second_completion = get_completion(
    TESTING_LIBRARY_NAME, 
    temperature=1.5
).choices[0].message.content


In [11]:
print(first_completion)
print(second_completion)

Certainly! Here are a few creative names for a new Python library:

1. PyroScope
2. FluxPy
3. Pythia
4. Pyrolysis
5. Pyrolyse
6. Pyrocode
7. Pyroplex
8. Pyrolyze
9. Pyrolytic
10. Pyrolyzea
11. Pyrolyticus
12. Pyrolyticus
13. Pyrolyticus
14. Pyrolyticus
15. Pyrolyticus

Feel free to choose one that resonates with your project's theme or functionality!
Certainly! Here are a few creative name suggestions for a new Python library:

1. PyPulse
2. Fluxify
3. SynapseKit
4. Zenith
5. Sparkle
6. QuantumKit
7. Meldify
8. Nexus
9. Sparkle
10. Pixelate

Feel free to share your thoughts on these, or let me know if you need more suggestions!


## `max_tokens` - The Safety Brake

`max_tokens` sets a hard limit on the number of tokens the model can generate. This is crucial for controlling costs and preventing overly long responses.

Let's see what happens when we ask for a summary with a very restrictive limit.

In [14]:
SUMMARY_PROMPT = [
    {
        "role": "user",
        "content": "Summarize the concept of OOP in two sentences"
    }
]

short_completion = get_completion(
    SUMMARY_PROMPT,
    max_tokens=10
)

long_completion = get_completion(
    SUMMARY_PROMPT,
    max_tokens=100
)
 

In [16]:
print(short_completion.choices[0].message.content)
print(long_completion.choices[0].message.content)
print(f"\nTotal tokens: {long_completion.usage.total_tokens}") 

Object-Oriented Programming (OOP) is
Object-Oriented Programming (OOP) is a programming paradigm based on the concept of "objects", which can contain data in the form of fields, or attributes, and code in the form of procedures, or methods. It allows for the creation of objects that can interact with each other to solve complex problems.

Total tokens: 108


## `stop` - The Clean Ending

The `stop` parameter tells the model to stop generating as soon as it encounters a specific sequence of characters. This is perfect for generating lists or other structured data where you want to prevent extra conversational text.

Let's ask for a list of programming paradigms and stop after the first item.

In [18]:
LIST_PROMPT = [
    {
        "role": "user",
        "content": "List the top 3 programming paradigms. Start with '1.'."
    }
]

no_stop_parameter = get_completion(
    LIST_PROMPT
)

with_stop_parameter = get_completion(
    LIST_PROMPT,
    stop=["\n3."]
)


In [19]:
print ("--- without stop parameter ===")
print (no_stop_parameter.choices[0].message.content)
print ("--- with stop parameter ===")
print (with_stop_parameter.choices[0].message.content)


--- without stop parameter ===
1. Object-Oriented Programming (OOP)
2. Procedural Programming
3. Functional Programming
--- with stop parameter ===
1. Object-oriented programming (OOP)
2. Functional programming


## `n` - Generating Multiple Choices

The `n` parameter lets you get multiple different responses for the same prompt in a single API call. This is great for brainstorming when combined with a higher temperature.

In [20]:
SLOGAN_PROMPT = [
    {
        "role": "user",
        "content": "Write a marketing slogan for my newly created AI-powered debug tool"
    }
]

response_n_choices = get_completion(
    SLOGAN_PROMPT,
    temperature=1.5,
    n=3
)
# note: qwen2.5 does not support n so will not run this any more 

UnsupportedParamsError: litellm.UnsupportedParamsError: ollama does not support parameters: ['n'], for model=qwen2.5:3b. To drop these, set `litellm.drop_params=True` or for proxy:

`litellm_settings:
 drop_params: true`
. 
 If you want to use these params dynamically send allowed_openai_params=['n'] in your request.

## `response_format` - Guaranteed JSON

For developers, this is a game-changer. By setting `response_format={"type": "json_object"}`, you can force the model to output a syntactically correct JSON object. This is only available on newer models like `gpt-4o-mini` and is incredibly reliable.

Let's extract information from a sentence into a structured JSON object.

In [21]:
JSON_PROMPT = [
    {
        "role": "user",
        "content": "Extract a JSON object with the user's name, city, and product they are asking about"
    },
    {
        "role": "user",
        "content": "Hi, this is Alex from Berlin. I have a question about the 'Quantum03' server"
    }
]

normal_completion = get_completion(
    JSON_PROMPT
)

json_completion = get_completion(
    JSON_PROMPT,
    response_format={"type": "json_object"}
)


In [23]:
def parse_json(json_str):
    try:
        parsed_json = json.loads(json_str)
        print("\nSuccessfully parsed JSON:")
        print(parsed_json)
    except json.JSONDecodeError as e:
        print(f"\nFailed to parse JSON: {e}")

parse_json(normal_completion.choices[0].message.content)
parse_json(json_completion.choices[0].message.content)

    


Failed to parse JSON: Expecting value: line 1 column 1 (char 0)

Successfully parsed JSON:
{'name': 'Alex', 'city': 'Berlin', 'product': 'Quantum03'}


In [24]:
print(normal_completion.choices[0].message.content)
print(json_completion.choices[0].message.content)


```json
{
  "name": "Alex",
  "city": "Berlin",
  "product": "Quantum03"
}
```
{"name": "Alex", "city": "Berlin", "product": "Quantum03"}
